# Chapter 4: Production Reliability

Estimated time: ~7 hours.

Prerequisites: Chapter 1 (`agentlib.llm_client`, the real/mock toggle, reused for this
chapter's real rate-limit section).

Interview category this chapter maps to: production reliability patterns, retry vs.
circuit breaker vs. cache invalidation, and "walk me through how you'd debug context
freshness in production," a very common systems-judgment question for anything built on top
of an LLM with retrieved or cached context.

## Concept: what changes between demo and production

A demo runs once, with curated input, on your machine, with nobody else hitting it at the
same time. Production runs continuously, with input you don't control, under concurrent
load, against dependencies that sometimes fail. It also keeps running long enough for state
you cached five minutes ago to quietly go stale. Nothing about the *model* changes between
demo and production; everything about the *environment around it* does.

Three patterns this chapter builds hands-on:

| Pattern | What it's for | Employee framing |
|---|---|---|
| Cache invalidation / TTL | Serving fresh data without re-fetching everything on every request | An employee working from a printout instead of pulling up the latest version. That's fine, as long as someone throws the printout away the moment the source changes |
| Retry with backoff + jitter | Recovering automatically from a *transient* failure (a flaky network call, a momentary overload) | Trying again after a dependency hiccups, but waiting a bit longer each time instead of hammering it immediately |
| Circuit breaker | Stopping work against a dependency that's *not* transiently failing, it's just down | Noticing a vendor hasn't answered the phone in three tries and stopping calling for a while, instead of redialing forever |

Staleness is the gap between what your cache says and what's actually true right now.
TTL (time-to-live) is a cache's built-in staleness budget: how long an entry is trusted
before it's treated as expired, regardless of whether it's actually still correct.
Graceful degradation is what a system does when a dependency fails and it *can't* fully
recover: it falls back to something reduced but useful (a stale-but-labeled answer, a
simpler response) rather than failing outright.

## Setup

In [1]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import random
import time

from agentlib.grading import check
from agentlib import llm_client

random.seed(42)
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")


LLM_PROVIDER = 'anthropic', HAS_KEY = False


## Build: a fake mini-codebase, with a version history

A small codebase (4 files, realistic docstrings) plus a version history per file: a
list of `(timestamp, content)` pairs simulating real edits over time. This is what an "AI
coding assistant" would be indexing in production, and it's the thing that goes stale.

In [2]:
CODEBASE_HISTORY = {
    "billing.py": [
        (0, '''def calculate_total(items):
    """Sum the price of every item in the cart. No discount support."""
    return sum(item["price"] for item in items)
'''),
        (3600, '''def calculate_total(items, discount_code=None):
    """Sum the price of every item in the cart, applying a discount code if provided."""
    total = sum(item["price"] for item in items)
    if discount_code == "SAVE10":
        total *= 0.9
    return total
'''),
    ],
    "auth.py": [
        (0, '''def authenticate(username, password):
    """Check a username/password pair against the user table. No rate limiting."""
    return _lookup_user(username) and _check_password(username, password)
'''),
    ],
    "notifications.py": [
        (0, '''def send_email(to, subject, body):
    """Send a transactional email via the configured provider."""
    return _provider.send(to=to, subject=subject, body=body)
'''),
    ],
    "inventory.py": [
        (0, '''def reserve_stock(sku, quantity):
    """Reserve stock for an order. Raises InsufficientStockError if unavailable."""
    if _available(sku) < quantity:
        raise InsufficientStockError(sku)
    _decrement(sku, quantity)
'''),
    ],
}


class FileStore:
    """Simulates a version-controlled codebase: get_file(name, at_time) returns whatever
    content was current at that timestamp."""

    def __init__(self, history: dict):
        self.history = history

    def get_file(self, filename: str, at_time: float) -> str:
        current = None
        for ts, content in self.history[filename]:
            if ts <= at_time:
                current = content
        return current


filestore = FileStore(CODEBASE_HISTORY)
print(filestore.get_file("billing.py", at_time=0))


def calculate_total(items):
    """Sum the price of every item in the cart. No discount support."""
    return sum(item["price"] for item in items)



### A caching layer with TTL

The cache is yours to build. It is a dictionary with an expiry clock, and the only decision
that matters is *when* the clock gets consulted.

Cache an answer at 9am with a two-hour TTL, then read it at 3pm. Nothing was written in
between, so nothing prompted the cache to reconsider anything. If expiry is only evaluated
at write time, that 9am answer is still sitting there at 3pm, six hours stale, and the
assistant serves it with total confidence. Staleness is a property of the read.

In [ ]:
class TTLCache:
    '''A cache whose entries go stale on a clock.

    There is no wall clock here: every method takes `now` explicitly, which is what makes
    the staleness behaviour testable without sleeping through a real TTL.

    - get(key, now) -> (value, cached_at) if the entry exists and is younger than ttl
                       seconds, else (None, None). An entry that has reached exactly ttl
                       seconds is expired.
    - set(key, value, now) -> store the value, stamped with `now`. Rewriting a key restarts
                       its clock.
    - invalidate(key) -> drop the entry; a key that was never cached is not an error.
    '''

    def __init__(self, ttl_seconds: float):
        self.ttl = ttl_seconds
        self.store = {}  # key -> (value, cached_at)

    def get(self, key, now: float):
        raise NotImplementedError("Implement me, then re-run this cell")

    def set(self, key, value, now: float):
        raise NotImplementedError("Implement me, then re-run this cell")

    def invalidate(self, key):
        raise NotImplementedError("Implement me, then re-run this cell")


TTLCache = check("ch04-ttl-cache", TTLCache)

In [4]:
cache = TTLCache(ttl_seconds=7200)  # 2-hour TTL
value, cached_at = cache.get("billing.py", now=0)
print("Before anything is cached:", value, cached_at)
cache.set("billing.py", filestore.get_file("billing.py", at_time=0), now=0)
value, cached_at = cache.get("billing.py", now=10)
print("After caching, a quick re-check:", value[:40].strip(), "...", "cached_at =", cached_at)

Before anything is cached: None None
After caching, a quick re-check: def calculate_total(items):
    """Sum t ... cached_at = 0


## Break it #1: the AI coding assistant works from an outdated printout

`billing.py` gets edited at t=3600 (discount code support is added). The cache's TTL is
7200 seconds, so if someone asks about `billing.py` again before the cache naturally
expires, they get the pre-edit answer: the code changed, but nothing told the cache.

In [5]:
def ai_assistant_answer_v1(filestore, cache, filename, question, now):
    '''Buggy version: no invalidation on write, and no logging of how old the served
    content actually is -- staleness is silent.'''
    cached_content, _ = cache.get(filename, now)
    if cached_content is not None:
        content = cached_content
    else:
        content = filestore.get_file(filename, now)
        cache.set(filename, content, now)
    has_discount = "discount_code" in content
    verdict = "supports" if has_discount else "does NOT support"
    return f"{filename} {verdict} discount codes."


buggy_cache = TTLCache(ttl_seconds=7200)

print("--- Bug: file is edited, but the cache doesn't know ---\n")
print("t=0:    ", ai_assistant_answer_v1(filestore, buggy_cache, "billing.py", "does it support discounts?", now=0))
print("        (billing.py is edited at t=3600 to add discount support -- the cache is never told)")
print("t=3700: ", ai_assistant_answer_v1(filestore, buggy_cache, "billing.py", "does it support discounts?", now=3700))
print("\nThe file has supported discount codes for 100 seconds by t=3700, but the assistant")
print("still says it doesn't -- and nothing in the output above tells you the answer is stale.")


--- Bug: file is edited, but the cache doesn't know ---

t=0:     billing.py does NOT support discount codes.
        (billing.py is edited at t=3600 to add discount support -- the cache is never told)
t=3700:  billing.py does NOT support discount codes.

The file has supported discount codes for 100 seconds by t=3700, but the assistant
still says it doesn't -- and nothing in the output above tells you the answer is stale.


In [6]:
def ai_assistant_answer_v2(filestore, cache, filename, question, now, log):
    '''Fixed version: logs context age on EVERY response (so staleness is detectable even
    if it happens), and the codebase-edit event below invalidates the cache on write (so it
    mostly doesn't happen in the first place). Two-layer fix, not one.'''
    cached_content, cached_at = cache.get(filename, now)
    if cached_content is not None:
        content = cached_content
        context_age = now - cached_at
        source = "cache"
    else:
        content = filestore.get_file(filename, now)
        cache.set(filename, content, now)
        context_age = 0
        source = "live fetch"

    log.append({"filename": filename, "source": source, "context_age_seconds": context_age})

    has_discount = "discount_code" in content
    verdict = "supports" if has_discount else "does NOT support"
    return f"{filename} {verdict} discount codes. [context age: {context_age}s, source: {source}]"


def edit_file(cache, filename):
    '''What a real deploy/webhook would trigger: invalidate the cache entry the moment the
    underlying file changes, instead of waiting for TTL to catch up.'''
    cache.invalidate(filename)


fixed_cache = TTLCache(ttl_seconds=7200)
freshness_log = []

print("--- Fix: invalidate on write, log context age on every response ---\n")
print("t=0:    ", ai_assistant_answer_v2(filestore, fixed_cache, "billing.py", "does it support discounts?", now=0, log=freshness_log))
edit_file(fixed_cache, "billing.py")
print("        (billing.py is edited at t=3600 -- this time the cache is invalidated immediately)")
print("t=3700: ", ai_assistant_answer_v2(filestore, fixed_cache, "billing.py", "does it support discounts?", now=3700, log=freshness_log))

print("\nFreshness log (this is what you'd actually check in production):")
for entry in freshness_log:
    print(" ", entry)


--- Fix: invalidate on write, log context age on every response ---

t=0:     billing.py does NOT support discount codes. [context age: 0s, source: live fetch]
        (billing.py is edited at t=3600 -- this time the cache is invalidated immediately)
t=3700:  billing.py supports discount codes. [context age: 0s, source: live fetch]

Freshness log (this is what you'd actually check in production):
  {'filename': 'billing.py', 'source': 'live fetch', 'context_age_seconds': 0}
  {'filename': 'billing.py', 'source': 'live fetch', 'context_age_seconds': 0}


## Break it #2: a tool that fails intermittently

Not every failure is permanent. A flaky dependency that fails *some* of the time is exactly
what retries exist for, as long as you back off between attempts instead of hammering it
immediately, and add jitter so many clients retrying at once don't all land on the same
retry schedule.

In [7]:
def make_flaky_tool(fail_probability: float, seed: int):
    rng = random.Random(seed)

    def flaky_tool():
        if rng.random() < fail_probability:
            raise ConnectionError("simulated transient failure")
        return "success"

    return flaky_tool


print("--- Bug: no retry, one failure ends the task ---\n")
flaky = make_flaky_tool(fail_probability=0.7, seed=1)
try:
    result = flaky()
    print("Result:", result)
except Exception as exc:
    print(f"Failed on the first attempt: {exc}")
    print("A 70%-fail-rate dependency will kill most single-shot calls to it -- unacceptable")
    print("for anything an agent needs to complete reliably.")


--- Bug: no retry, one failure ends the task ---

Failed on the first attempt: simulated transient failure
A 70%-fail-rate dependency will kill most single-shot calls to it -- unacceptable
for anything an agent needs to complete reliably.


In [ ]:
def retry_with_backoff(fn, max_retries: int = 6, base_delay: float = 1.0, jitter: float = 0.5,
                        seed: int = 1, sleep_fn=None, retry_predicate=None):
    '''Reusable exponential backoff + jitter wrapper.

    Returns a `wrapped(*args, **kwargs)` that forwards through to fn and returns
    (result, attempts_log). Each failed attempt appends
    {"attempt": n, "error": str(exc), "backoff_s": round(delay, 2)} to that log.

    The delay before retry number `attempt` (1-based) is:

        base_delay * 2 ** (attempt - 1) + rng.uniform(0, jitter)

    Exponential, not linear. Linear growth looks almost identical over three attempts and
    then stops helping precisely when it matters: if a dependency is overloaded, backing off
    by a constant increment never outpaces the queue building up in front of it.

    Back off AFTER a failure. A call that succeeds first time must not sleep at all.

    sleep_fn defaults to None, which logs the delay each attempt WOULD have taken without
    actually sleeping (keeps this notebook fast); pass sleep_fn=time.sleep for real behavior
    against a real dependency. Use random.Random(seed) so runs are reproducible.

    retry_predicate, if given, is called with the exception; a False verdict re-raises
    immediately instead of retrying (see the next cell). Exhausting max_retries raises
    RuntimeError `from` the last exception.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


retry_with_backoff = check("ch04-backoff", retry_with_backoff)

In [9]:
print("--- Fix: retry with exponential backoff + jitter ---\n")
robust_flaky = retry_with_backoff(make_flaky_tool(fail_probability=0.7, seed=1), max_retries=8)
result, attempts_log = robust_flaky()
for a in attempts_log:
    print(f"  attempt {a['attempt']} failed ({a['error']}) -- backed off {a['backoff_s']}s before retrying")
print(f"\nResult: {result!r}, succeeded on attempt {len(attempts_log) + 1}")

--- Fix: retry with exponential backoff + jitter ---

  attempt 1 failed (simulated transient failure) -- backed off 1.07s before retrying

Result: 'success', succeeded on attempt 2


### Not everything is worth retrying

Backoff answers *how long to wait*. It doesn't answer *whether to wait at all*, and applying
it indiscriminately is its own bug: a malformed request body is exactly as malformed on the
sixth attempt as the first, so six exponential backoffs just add half a minute of latency
before the caller finally sees the error they could have had immediately.

The split is not simply "4xx bad, 5xx good". Two 4xx statuses are genuinely transient -- 429
(rate limited) and 408 (request timeout) -- and they are the two that most need a retry.
Treating the whole 4xx range as non-retryable throws both away.

In [ ]:
class ApiError(Exception):
    '''Stands in for a provider SDK's HTTP error, which carries a status code.'''

    def __init__(self, status_code, message="api error"):
        super().__init__(f"{status_code}: {message}")
        self.status_code = status_code


def should_retry(error: Exception) -> bool:
    '''Is retrying this error worth anything, or will it fail identically every time?

    Retry: transport failures with no status at all (ConnectionError, TimeoutError), the
    5xx range, and the two transient 4xx statuses, 408 and 429.

    Don't retry: the rest of the 4xx range -- 400, 401, 403, 404, 422 all describe a request
    that is broken in a way another attempt cannot fix.

    An error carrying no status_code at all is treated as transient.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


should_retry = check("ch04-no-retry-4xx", should_retry)

In [11]:
def make_status_tool(status_code):
    def tool():
        raise ApiError(status_code)
    return tool


print("--- The same wrapper, now asking whether the error is worth retrying at all ---\n")
for status in (429, 400):
    attempted = {"n": 0}

    def counted():
        attempted["n"] += 1
        make_status_tool(status)()

    guarded = retry_with_backoff(counted, max_retries=5, retry_predicate=should_retry)
    try:
        guarded()
    except Exception as exc:
        print(f"  HTTP {status}: gave up after {attempted['n']} attempt(s) -- {type(exc).__name__}")

print()
print("A 429 is worth five attempts; a 400 is worth exactly one. Without the predicate both")
print("cost the same five attempts and the same total backoff, and the 400's caller waits")
print("through all of it to be told something the very first response already said.")

--- The same wrapper, now asking whether the error is worth retrying at all ---

  HTTP 429: gave up after 5 attempt(s) -- RuntimeError
  HTTP 400: gave up after 1 attempt(s) -- ApiError

A 429 is worth five attempts; a 400 is worth exactly one. Without the predicate both
cost the same five attempts and the same total backoff, and the 400's caller waits
through all of it to be told something the very first response already said.


## Break it #3: a cascading failure

Retries are the wrong tool when a dependency isn't flaky, it's just down. Retrying a
fully-dead dependency doesn't help it recover; it just adds load to an already-failing
system and wastes time on the caller's side, potentially cascading the failure to whatever
*is* calling you. A circuit breaker (Nygard, 2007; see `REFERENCES.md`) stops that: after
enough consecutive failures it "opens" and fails fast (no call is even attempted) until a
cooldown period passes and it allows one trial call through.

In [12]:
def make_always_failing_tool():
    def broken_tool():
        raise ConnectionError("dependency is down")
    return broken_tool


print("--- Bug: naive caller keeps hammering a fully-dead dependency ---\n")
broken = make_always_failing_tool()
attempted = 0
for t in range(10):
    try:
        broken()
    except Exception:
        pass
    attempted += 1
print(f"Made {attempted} calls to a dependency that never once succeeded.")
print("Every one of those wastes time on both sides -- a real network call here would add")
print("real latency to each failure, and do it 10 times over for zero benefit.")


--- Bug: naive caller keeps hammering a fully-dead dependency ---

Made 10 calls to a dependency that never once succeeded.
Every one of those wastes time on both sides -- a real network call here would add
real latency to each failure, and do it 10 times over for zero benefit.


In [ ]:
class CircuitBreaker:
    '''States: closed (normal) -> open (failing fast, not calling the dependency at all)
    -> half-open (after a cooldown, allow one trial call through) -> closed on success,
    back to open on failure.

    call(fn, now) is the whole interface. While closed, it calls fn, counts CONSECUTIVE
    failures (a success resets the count), and opens once the count reaches
    failure_threshold, recording `now` as opened_at. While open, it raises immediately
    WITHOUT calling fn -- that is the entire saving.

    The half-open transition is the part that is easy to leave out and impossible to notice
    you left out, because everything looks correct right up until the dependency recovers.
    Once `cooldown` seconds have elapsed since opened_at, exactly one trial call must be let
    through. Succeed and the circuit closes; fail and it opens again with a fresh cooldown.
    Without it the breaker never re-tests anything, and the outage outlives its own cause.

    Failures reaching fn still re-raise, so the caller sees the real error.
    '''

    def __init__(self, failure_threshold: int = 3, cooldown: float = 5):
        self.failure_threshold = failure_threshold
        self.cooldown = cooldown
        self.failure_count = 0
        self.state = "closed"
        self.opened_at = None

    def call(self, fn, now: float):
        raise NotImplementedError("Implement me, then re-run this cell")


CircuitBreaker = check("ch04-circuit-breaker", CircuitBreaker)

In [14]:
breaker = CircuitBreaker(failure_threshold=3, cooldown=5)
underlying_calls = 0


def counted_broken():
    global underlying_calls
    underlying_calls += 1
    return make_always_failing_tool()()


print("--- Fix: circuit breaker fails fast instead of hammering a dead dependency ---\n")
for t in range(10):
    try:
        breaker.call(counted_broken, now=t)
    except Exception as exc:
        print(f"t={t}: {exc} (breaker state: {breaker.state})")

print(f"\nUnderlying dependency was actually invoked {underlying_calls} times out of 10 attempts")
print("(vs. 10/10 in the buggy version above) -- the breaker opened after 3 failures and")
print("stopped calling the dependency at all for the rest of the window.")

--- Fix: circuit breaker fails fast instead of hammering a dead dependency ---

t=0: dependency is down (breaker state: closed)
t=1: dependency is down (breaker state: closed)
t=2: dependency is down (breaker state: open)
t=3: circuit open -- failing fast (dependency not called) (breaker state: open)
t=4: circuit open -- failing fast (dependency not called) (breaker state: open)
t=5: circuit open -- failing fast (dependency not called) (breaker state: open)
t=6: circuit open -- failing fast (dependency not called) (breaker state: open)
t=7: dependency is down (breaker state: open)
t=8: circuit open -- failing fast (dependency not called) (breaker state: open)
t=9: circuit open -- failing fast (dependency not called) (breaker state: open)

Underlying dependency was actually invoked 4 times out of 10 attempts
(vs. 10/10 in the buggy version above) -- the breaker opened after 3 failures and
stopped calling the dependency at all for the rest of the window.


In [15]:
print("--- Recovery: the dependency comes back up, the breaker notices via one trial call ---\n")

recovery_calls = 0


def working_tool():
    global recovery_calls
    recovery_calls += 1
    return "success"


recovery_time = breaker.opened_at + breaker.cooldown + 1
result = breaker.call(working_tool, now=recovery_time)
print(f"t={recovery_time}: call succeeded ({result!r}), breaker state is now {breaker.state!r}")
print(f"Only {recovery_calls} trial call was needed to detect recovery -- not a burst of retries.")


--- Recovery: the dependency comes back up, the breaker notices via one trial call ---

t=13: call succeeded ('success'), breaker state is now 'closed'
Only 1 trial call was needed to detect recovery -- not a burst of retries.


## Real rate-limit handling

Reusing `agentlib.llm_client` from Chapter 1: with a real key present, fire a burst of
concurrent calls large enough to actually trigger a real `429`/overloaded response from
whichever provider is active, then apply the exact same `retry_with_backoff()` built above
against the real API: real timing, real errors, not a simulation. Falls back to a second
run of the simulated flaky-tool exercise if no key is present.

In [16]:
if llm_client.HAS_KEY:
    import concurrent.futures

    def burst_call(i):
        return llm_client.call_model(
            messages=[{"role": "user", "content": f"Reply with only the number {i}."}],
            model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
            max_tokens=10,
        )

    print("Firing 20 concurrent real API calls to try to trigger a real rate limit...\n")
    errors, successes = [], []
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        futures = {executor.submit(burst_call, i): i for i in range(20)}
        for future in concurrent.futures.as_completed(futures):
            try:
                successes.append(future.result())
            except Exception as exc:
                errors.append(exc)

    print(f"{len(successes)} succeeded, {len(errors)} hit an error (rate limit or otherwise) on the first attempt.")
    if errors:
        print(f"Example real error: {errors[0]!r}")

    print("\nApplying retry_with_backoff() against the real API for one call:")
    robust_call = retry_with_backoff(lambda: burst_call(0), max_retries=5, sleep_fn=time.sleep)
    result, attempts_log = robust_call()
    for a in attempts_log:
        print(f"  attempt {a['attempt']} failed ({a['error']}) -- backed off {a['backoff_s']}s")
    print(f"Final result text: {result.text!r}")
else:
    print("No API key present -- reusing the simulated flaky-tool exercise above as the")
    print("CI/no-budget path instead of firing real concurrent calls.\n")
    simulated_flaky = make_flaky_tool(fail_probability=0.6, seed=7)
    robust_simulated = retry_with_backoff(simulated_flaky, max_retries=6)
    result, attempts_log = robust_simulated()
    for a in attempts_log:
        print(f"  attempt {a['attempt']} failed ({a['error']}) -- backed off {a['backoff_s']}s")
    print(f"Result: {result!r}, succeeded on attempt {len(attempts_log) + 1}")


No API key present -- reusing the simulated flaky-tool exercise above as the
CI/no-budget path instead of firing real concurrent calls.

  attempt 1 failed (simulated transient failure) -- backed off 1.07s
  attempt 2 failed (simulated transient failure) -- backed off 2.42s
Result: 'success', succeeded on attempt 3


## Interview preparation

### Recap

- A demo and a production system differ in load, concurrency, and time, not in the model.
  Most "AI reliability" problems are ordinary distributed-systems problems wearing an LLM
  costume.
- Staleness is silent unless you make it visible. This chapter's fix logged context age on
  every response specifically so a stale answer is detectable instead of just wrong.
- Retries, circuit breakers, and cache invalidation each answer a different failure, and
  reaching for the wrong one has its own specific cost. The recall drill below is where you
  work out which is which; this chapter built all three so you have something to reason
  from.

### Debug this from the actual logs

"Users complain the AI coding assistant ignores recent code changes." Walk through how you'd
debug context freshness in production, using the `freshness_log` this chapter actually
generated above, not from memory. What would you check first, and what in that log format
specifically would tell you whether this is a caching bug (like this chapter's break-it #1)
versus something else entirely (e.g. the index genuinely hasn't re-ingested the file yet)?

### Reliability-pattern recall drill

For each scenario, name the pattern (retry / circuit breaker / cache invalidation) that
actually applies, and, just as important, say what happens if you reach for the wrong one:

1. A downstream service is fully down for the next 20 minutes during a deploy.
2. A network call fails about 1 in 20 times with no discernible pattern.
3. A document was updated five minutes ago, but an agent is still citing the old version.
4. A downstream service returns errors for 30 seconds during a brief traffic spike, then
   fully recovers on its own.

Check your answers against `solutions/ch04_production_reliability_answers.md`.

#### Answering these

There is a slot below for each question. Write your answer into it, run the cell, then use
`drill.check(n)` to see your answer and the model answer side by side.

`check(n)` will not show you an answer until you have written one of your own — once you have
read the model answer you can no longer find out what you actually knew. If you want it
anyway, `drill.reveal(n)` is there and makes no judgement.

Answers are read from `solutions/ch04_*_answers.md` at runtime, so nothing in this
notebook contains one.

In [17]:
from agentlib.self_check import drill as open_drill

drill = open_drill(4)
drill.questions()

Chapter 4 written drill — 4 questions

1. A downstream service is fully down for the next 20 minutes during a deploy.
2. A network call fails about 1 in 20 times with no discernible pattern.
3. A document was updated five minutes ago, but an agent is still citing the old version.
4. A downstream service returns errors for 30 seconds during a brief traffic spike, then fully
   recovers on its own.


In [18]:
# One slot per question. Replace the placeholder text, then run this cell.
# Anything under 25 words, or left as the placeholder, is not recorded.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

print()
drill.status()

  1. not recorded — it is still the placeholder
  2. not recorded — it is still the placeholder
  3. not recorded — it is still the placeholder
  4. not recorded — it is still the placeholder

Chapter 4: 0/4 answered
  still open: [1, 2, 3, 4]


In [19]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

Question 1 has no recorded answer yet.

  A downstream service is fully down for the next 20 minutes during a deploy.

Write one with attempt() first. Reading the model answer before you have committed to your own turns this into a reading exercise -- once you have seen it you can no longer find out what you actually knew.
(If you really want it anyway: reveal(1).)


## Next up: Chapter 5, Cost, Performance, and Model Selection

This chapter was about surviving failures. Chapter 5 is about the cost of *not* failing:
token economics, latency decomposition, and matching model choice to task complexity, so
you're not paying senior-employee rates for junior-employee work.